# RRSIS-D Semantic Segmentation — SegFormer v2

**Model**: `nvidia/mit-b2` (SegFormer-B2) fine-tuned from ADE20K  
**Task**: 20-class remote sensing image segmentation  
**Dataset**: RRSIS-D (17,402 XML-annotated images)  

### What's in this notebook
- Image size: **480 × 480**
- Class-imbalance handling via weighted Focal + Dice loss + WeightedRandomSampler
- Albumentations augmentation pipeline (remote-sensing aware)
- **Early stopping** (patience-based on val mIoU)
- **LR scheduler**: OneCycleLR (per-batch) + ReduceLROnPlateau (per-epoch fallback)
- **Full metrics**: Accuracy, Precision, Recall, F1, Confusion Matrix, mIoU, oIoU, Dice
- **Model info**: size on disk, total/trainable parameters, training time per epoch
- **Training curves**: Loss, Accuracy, mIoU vs epochs (train + val)
- Optuna hyperparameter tuning
- Mixed precision training (FP16)

> **Setup**: Runtime → Change runtime type → T4 GPU (free) or A100 (Colab Pro)

---
## 1. Install Dependencies

In [ ]:
# transformers  : HuggingFace SegFormer model
# albumentations: best-in-class image augmentation library
# optuna        : Bayesian hyperparameter tuning (no server needed)
# scikit-learn  : precision / recall / F1 / confusion matrix utilities
# seaborn       : heatmap for confusion matrix visualisation
!pip install -q transformers albumentations optuna timm evaluate datasets accelerate
!pip install -q scikit-learn seaborn

---
## 2. Mount Google Drive & Set Paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ─────────────────────────────────────────────────────────────
# UPDATE these paths to where your data lives on Google Drive.
# Example layout:
#   MyDrive/RRSIS-D/JPEGImages/   ← all .jpg images
#   MyDrive/RRSIS-D/ann_split/    ← all .xml annotations
# ─────────────────────────────────────────────────────────────
IMAGE_DIR  = '/content/drive/MyDrive/RRSIS-D/JPEGImages'
LABEL_DIR  = '/content/drive/MyDrive/RRSIS-D/ann_split'
OUTPUT_DIR = '/content/drive/MyDrive/RRSIS-D/checkpoints'

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Images : {IMAGE_DIR}')
print(f'Labels : {LABEL_DIR}')
print(f'Outputs: {OUTPUT_DIR}')

---
## 3. Core Imports

In [ ]:
import os, glob, json, random, time, xml.etree.ElementTree as ET
from pathlib import Path
from collections import Counter

import numpy as np
import cv2
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.cuda.amp import GradScaler, autocast

import albumentations as A
from albumentations.pytorch import ToTensorV2

from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    accuracy_score, confusion_matrix, classification_report
)

from transformers import (
    SegformerForSemanticSegmentation,
    SegformerImageProcessor,
)
import optuna
from optuna.samplers import TPESampler
optuna.logging.set_verbosity(optuna.logging.WARNING)  # suppress noisy trial logs

# ── Reproducibility ──────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU   : {torch.cuda.get_device_name(0)}')
    print(f'VRAM  : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

---
## 4. Class Definitions & Label Mapping

Class index **0 = background**; object classes occupy indices 1–20.
This is required by SegFormer's config and by the weighted loss.

In [ ]:
CLASSES = [
    'Expressway-Service-area', 'Expressway-toll-station', 'airplane',
    'airport', 'baseballfield', 'basketballcourt', 'bridge', 'chimney',
    'dam', 'golffield', 'groundtrackfield', 'harbor', 'overpass', 'ship',
    'stadium', 'storagetank', 'tenniscourt', 'trainstation', 'vehicle',
    'windmill'
]

# Background (0) + 20 object classes → 21 total labels
ALL_LABELS = ['background'] + CLASSES
NUM_LABELS = len(ALL_LABELS)   # 21

label2id = {name: idx for idx, name in enumerate(ALL_LABELS)}
id2label = {idx: name for idx, name in enumerate(ALL_LABELS)}

# Object-class counts from dataset analysis (indices 0–19 = CLASSES[0–19])
CLASS_COUNTS = [
    1417, 1102, 1688, 1241, 3658, 1287, 2486, 1243,
    1037, 1032, 2455,  583, 2234, 1960, 1138, 1255,
    1519,  891, 7098, 2996
]

print(f'Total labels (incl. background): {NUM_LABELS}')

---
## 5. Class Imbalance — Inverse-Frequency Loss Weights

Imbalance ratio is **12.17×** (vehicle: 7098 vs harbor: 583).  
Formula: `w_c = total / (n_classes × count_c)`, then normalise so mean = 1.  
Background gets weight **0.1** to focus the model on foreground objects.

In [ ]:
def compute_class_weights(counts, bg_weight=0.1):
    """
    Inverse-frequency class weights.
    counts    : list of per-class object counts (len = num object classes)
    bg_weight : weight for class-0 (background)
    Returns FloatTensor of shape (num_labels,)
    """
    arr    = np.array(counts, dtype=np.float64)
    total  = arr.sum()
    n      = len(arr)
    w      = total / (n * arr)        # large weight = rare class
    w      = w / w.mean()             # normalise so mean == 1.0
    w      = np.concatenate([[bg_weight], w])   # prepend background weight
    return torch.tensor(w, dtype=torch.float32)


CLASS_WEIGHTS = compute_class_weights(CLASS_COUNTS)

print('Top-5 highest weights (rarest classes):')
pairs = sorted(zip(ALL_LABELS, CLASS_WEIGHTS.tolist()), key=lambda x: -x[1])
for name, w in pairs[:6]:
    print(f'  {name:<32s}  {w:.4f}')

---
## 6. XML Annotation Parser → Segmentation Mask

Each PASCAL-VOC XML file has `<object>` entries with `<name>` and `<bndbox>`.  
We paint bounding boxes onto a (H×W) uint8 mask, large→small so small objects
(e.g. vehicles, chimneys) are never buried under larger annotations.

In [ ]:
def parse_xml_to_mask(xml_path: str, img_h: int, img_w: int) -> np.ndarray:
    """
    Parse a PASCAL-VOC XML and return a uint8 segmentation mask.
    Pixel value = label index (0 = background).
    Objects are painted large-first so small objects sit on top.
    """
    mask = np.zeros((img_h, img_w), dtype=np.uint8)
    tree = ET.parse(xml_path)
    root = tree.getroot()

    objects = []
    for obj in root.findall('object'):
        name = obj.find('name').text.strip()
        if name not in label2id:
            continue
        bb   = obj.find('bndbox')
        xmin = max(0,     int(float(bb.find('xmin').text)))
        ymin = max(0,     int(float(bb.find('ymin').text)))
        xmax = min(img_w, int(float(bb.find('xmax').text)))
        ymax = min(img_h, int(float(bb.find('ymax').text)))
        area = (xmax - xmin) * (ymax - ymin)
        objects.append((area, xmin, ymin, xmax, ymax, label2id[name]))

    # Sort descending by area — paint large boxes first
    objects.sort(reverse=True, key=lambda x: x[0])
    for _, xmin, ymin, xmax, ymax, cls_idx in objects:
        mask[ymin:ymax, xmin:xmax] = cls_idx

    return mask


# ── Quick sanity check ────────────────────────────────────────
sample_xmls = sorted(glob.glob(os.path.join(LABEL_DIR, '**', '*.xml'), recursive=True))
print(f'Found {len(sample_xmls)} XML files')

if sample_xmls:
    sx       = sample_xmls[0]
    img_name = Path(sx).stem + '.jpg'
    img_path = os.path.join(IMAGE_DIR, img_name)
    if os.path.exists(img_path):
        _img = cv2.imread(img_path)
        _h, _w = _img.shape[:2]
        _mask  = parse_xml_to_mask(sx, _h, _w)
        unique = np.unique(_mask)
        print(f'Sample : {img_name}  size={_w}x{_h}')
        print(f'Classes: {[id2label[u] for u in unique]}')

---
## 7. Build File List & Train / Val / Test Split

In [ ]:
def build_file_pairs(image_dir, label_dir):
    """Return list of (img_path, xml_path) tuples for matched files."""
    xml_paths = sorted(glob.glob(os.path.join(label_dir, '**', '*.xml'), recursive=True))
    pairs = []
    for xp in xml_paths:
        img_p = os.path.join(image_dir, Path(xp).stem + '.jpg')
        if os.path.exists(img_p):
            pairs.append((img_p, xp))
    return pairs


all_pairs = build_file_pairs(IMAGE_DIR, LABEL_DIR)
print(f'Valid (image, xml) pairs: {len(all_pairs)}')

# ── 70 / 15 / 15 random split ─────────────────────────────────
random.shuffle(all_pairs)
n       = len(all_pairs)
n_val   = int(n * 0.15)
n_test  = int(n * 0.15)

test_pairs  = all_pairs[:n_test]
val_pairs   = all_pairs[n_test : n_test + n_val]
train_pairs = all_pairs[n_test + n_val :]

print(f'Train: {len(train_pairs)}  Val: {len(val_pairs)}  Test: {len(test_pairs)}')

---
## 8. Augmentation Pipelines (Albumentations)

**Image size changed to 480 × 480** as requested.

Remote-sensing–aware choices:
- All 4 flip/rotate variants valid (no canonical orientation in aerial images)
- `RandomShadow` → cloud/building shadow on ground
- `CLAHE` → atmospheric haze / low-contrast scenes
- `CoarseDropout` → occlusion robustness for small objects
- `Perspective` → off-nadir imaging simulation
- ImageNet normalisation (SegFormer-B2 is ImageNet-22K pretrained)

In [ ]:
IMG_SIZE = 480   # ← changed to 480×480 as requested

# ── Training transforms ───────────────────────────────────────
train_transforms = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),

    # Spatial augmentations — applied identically to image AND mask
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.RandomResizedCrop(
        size=(IMG_SIZE, IMG_SIZE), scale=(0.6, 1.0), ratio=(0.8, 1.2), p=0.4
    ),
    A.Perspective(scale=(0.02, 0.06), p=0.3),
    A.GridDistortion(num_steps=5, distort_limit=0.15, p=0.2),

    # Pixel-level augmentations — image only
    A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, p=0.6),
    A.CLAHE(clip_limit=4.0, p=0.3),               # local contrast enhancement
    A.HueSaturationValue(hue_shift_limit=10,
                         sat_shift_limit=20,
                         val_shift_limit=20, p=0.4),
    A.GaussianBlur(blur_limit=(3, 5), p=0.2),
    A.Sharpen(alpha=(0.1, 0.4), lightness=(0.8, 1.2), p=0.2),
    A.RandomShadow(shadow_roi=(0, 0, 1, 1),        # simulates cloud shadow
                   num_shadows_limit=(1, 2),
                   shadow_dimension=5, p=0.2),
    A.GaussNoise(std_range=(0.01, 0.05), p=0.2),
    A.CoarseDropout(                               # forces partial-view robustness
        num_holes_range=(1, 4),
        hole_height_range=(0.05, 0.15),
        hole_width_range=(0.05, 0.15),
        p=0.3
    ),

    # Normalise with ImageNet stats, then convert to tensor
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

# ── Validation / Test transforms (no augmentation) ────────────
val_transforms = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

print(f'Image size : {IMG_SIZE}×{IMG_SIZE}')
print('Transforms ready.')

---
## 9. Dataset Class

In [ ]:
class RRSISDataset(Dataset):
    """
    RRSIS-D PyTorch Dataset.
    __getitem__ returns:
      pixel_values : FloatTensor (3, H, W)  — normalised image
      labels       : LongTensor  (H, W)     — segmentation mask (0=bg)
    """

    def __init__(self, pairs, transforms=None):
        self.pairs      = pairs
        self.transforms = transforms

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        img_path, xml_path = self.pairs[idx]

        # Load image (BGR → RGB)
        img = cv2.imread(img_path)
        if img is None:
            raise FileNotFoundError(f'Image not found: {img_path}')
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h, w = img.shape[:2]

        # Build segmentation mask from XML
        mask = parse_xml_to_mask(xml_path, h, w)

        # Apply augmentations
        if self.transforms:
            out  = self.transforms(image=img, mask=mask)
            img  = out['image']   # Tensor (3, H, W)
            mask = out['mask']    # Tensor (H, W)

        if not isinstance(mask, torch.Tensor):
            mask = torch.from_numpy(mask)
        mask = mask.long()

        return {'pixel_values': img, 'labels': mask}

    def get_dominant_class(self, idx):
        """Most frequent foreground class index (for WeightedRandomSampler)."""
        img_path, xml_path = self.pairs[idx]
        img = cv2.imread(img_path)
        h, w = img.shape[:2]
        mask = parse_xml_to_mask(xml_path, h, w)
        fg   = mask[mask > 0]
        if len(fg) == 0:
            return 0
        return int(Counter(fg.tolist()).most_common(1)[0][0])


train_ds = RRSISDataset(train_pairs, transforms=train_transforms)
val_ds   = RRSISDataset(val_pairs,   transforms=val_transforms)
test_ds  = RRSISDataset(test_pairs,  transforms=val_transforms)

print(f'Train: {len(train_ds)}  Val: {len(val_ds)}  Test: {len(test_ds)}')

---
## 10. WeightedRandomSampler (Over-sample Rare Classes)

Each training image's sampling probability ∝ 1 / freq(dominant_class).  
Combined with the weighted loss, this is a two-level imbalance correction.

In [ ]:
SAMPLER_CACHE = os.path.join(OUTPUT_DIR, 'train_dominant_classes.npy')

if os.path.exists(SAMPLER_CACHE):
    dominant_classes = np.load(SAMPLER_CACHE)
    print(f'Loaded dominant-class cache ({len(dominant_classes)} entries).')
else:
    print('Computing dominant classes (runs once, then cached)...')
    dominant_classes = np.array(
        [train_ds.get_dominant_class(i) for i in range(len(train_ds))],
        dtype=np.int64
    )
    np.save(SAMPLER_CACHE, dominant_classes)
    print('Saved to cache.')

class_freq = np.bincount(dominant_classes, minlength=NUM_LABELS)
class_freq = np.where(class_freq == 0, 1, class_freq)
sample_weights = torch.tensor(
    [1.0 / class_freq[c] for c in dominant_classes], dtype=torch.float32
)

sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)
print('WeightedRandomSampler ready.')

---
## 11. DataLoaders

In [ ]:
BATCH_SIZE  = 8   # reduce to 4 if you hit CUDA OOM on T4
NUM_WORKERS = 2

# Train: uses WeightedRandomSampler (mutually exclusive with shuffle=True)
train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    sampler=sampler,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    drop_last=True,      # avoids BatchNorm crash on partial last batch
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

test_loader = DataLoader(
    test_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

print(f'Train batches: {len(train_loader)}')
print(f'Val   batches: {len(val_loader)}')
print(f'Test  batches: {len(test_loader)}')

---
## 12. Batch Visualisation Sanity Check

In [ ]:
CMAP = plt.cm.get_cmap('tab20', NUM_LABELS)

def denormalise(tensor):
    """Reverse ImageNet normalisation for display."""
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    return (tensor * std + mean).clamp(0, 1)


batch = next(iter(train_loader))
imgs, masks = batch['pixel_values'], batch['labels']

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for i in range(4):
    axes[0, i].imshow(denormalise(imgs[i]).permute(1, 2, 0).numpy())
    axes[0, i].set_title(f'Image [{i}]', fontsize=9)
    axes[0, i].axis('off')
    m = masks[i].numpy()
    axes[1, i].imshow(m, cmap=CMAP, vmin=0, vmax=NUM_LABELS - 1)
    labels_in = ', '.join(id2label[u][:8] for u in np.unique(m) if u > 0)[:35]
    axes[1, i].set_title(labels_in, fontsize=7)
    axes[1, i].axis('off')
plt.suptitle('Batch check — top: image  bottom: mask', y=1.01)
plt.tight_layout()
plt.show()
print(f'Image shape: {imgs.shape}  Mask shape: {masks.shape}')

---
## 13. Build SegFormer-B2 Model

In [ ]:
def build_segformer(num_labels=NUM_LABELS, label2id=label2id,
                    id2label=id2label, pretrained='nvidia/mit-b2'):
    """
    Load SegFormer-B2 pretrained on ADE20K and replace the decode head
    with a new head for our 21-class output.
    ignore_mismatched_sizes=True silences the 150→21 class-count warning.
    """
    return SegformerForSemanticSegmentation.from_pretrained(
        pretrained,
        num_labels=num_labels,
        id2label=id2label,
        label2id=label2id,
        ignore_mismatched_sizes=True,
    )


model = build_segformer().to(DEVICE)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total parameters      : {total_params:,}')
print(f'Trainable parameters  : {trainable_params:,}')
print(f'Non-trainable params  : {total_params - trainable_params:,}')

---
## 14. Model Size on Disk

We save a temporary checkpoint and measure its file size in MB.
This reflects the actual storage footprint of deploying this model.

In [ ]:
import tempfile

def get_model_size_mb(model):
    """Save model state_dict to a temp file and return its size in MB."""
    with tempfile.NamedTemporaryFile(suffix='.pt', delete=False) as tmp:
        torch.save(model.state_dict(), tmp.name)
        size_mb = os.path.getsize(tmp.name) / (1024 ** 2)
    os.unlink(tmp.name)   # clean up
    return size_mb


model_size_mb = get_model_size_mb(model)
print(f'Model size on disk    : {model_size_mb:.1f} MB')
print(f'Total parameters      : {total_params:,}')
print(f'Trainable parameters  : {trainable_params:,}')

---
## 15. Loss Function — Weighted Focal + Dice

- **Focal Loss** (α-weighted): down-weights easy background, focuses on hard/rare pixels  
- **Dice Loss**: directly optimises the IoU metric — critical for small objects  
- Combined: `loss = focal + dice_weight × dice`

In [ ]:
class FocalLoss(nn.Module):
    """Multi-class alpha-weighted focal loss. ignore_index=-100 skips padded pixels."""
    def __init__(self, alpha=None, gamma=2.0, ignore_index=-100):
        super().__init__()
        self.alpha        = alpha
        self.gamma        = gamma
        self.ignore_index = ignore_index

    def forward(self, logits, targets):
        alpha = self.alpha.to(logits.device) if self.alpha is not None else None
        ce    = F.cross_entropy(
            logits, targets,
            weight=alpha,
            ignore_index=self.ignore_index,
            reduction='none'
        )  # (B, H, W)
        with torch.no_grad():
            pt = torch.exp(-ce)
        focal = (1 - pt) ** self.gamma * ce
        valid = (targets != self.ignore_index)
        return focal[valid].mean()


class DiceLoss(nn.Module):
    """Soft multi-class Dice loss. smooth=1 prevents division by zero."""
    def __init__(self, smooth=1.0, ignore_index=-100):
        super().__init__()
        self.smooth       = smooth
        self.ignore_index = ignore_index

    def forward(self, logits, targets):
        B, C, H, W = logits.shape
        probs       = logits.softmax(dim=1)
        valid_mask  = (targets != self.ignore_index)
        targets_c   = targets.clone()
        targets_c[~valid_mask] = 0
        targets_oh  = F.one_hot(targets_c, C).permute(0, 3, 1, 2).float()
        valid_mask  = valid_mask.unsqueeze(1).float()
        probs       = probs * valid_mask
        targets_oh  = targets_oh * valid_mask
        inter       = (probs * targets_oh).sum(dim=(0, 2, 3))
        union       = probs.sum(dim=(0, 2, 3)) + targets_oh.sum(dim=(0, 2, 3))
        dice_cls    = (2 * inter + self.smooth) / (union + self.smooth)
        return 1.0 - dice_cls.mean()


class CombinedLoss(nn.Module):
    def __init__(self, class_weights, dice_weight=0.5, gamma=2.0):
        super().__init__()
        self.focal       = FocalLoss(alpha=class_weights, gamma=gamma)
        self.dice        = DiceLoss()
        self.dice_weight = dice_weight

    def forward(self, logits, targets):
        return self.focal(logits, targets) + self.dice_weight * self.dice(logits, targets)


criterion = CombinedLoss(CLASS_WEIGHTS, dice_weight=0.5, gamma=2.0)
print('Loss: Weighted Focal + Dice ready.')

---
## 16. Comprehensive Metrics

We compute:

| Metric | Source | Description |
|--------|--------|-------------|
| **Pixel Accuracy** | confusion matrix | % of pixels correctly classified |
| **Precision** (macro) | sklearn | avg precision per class |
| **Recall** (macro) | sklearn | avg recall per class |
| **F1 Score** (macro) | sklearn | harmonic mean of P and R |
| **mIoU** | confusion matrix | mean IoU over all present classes |
| **oIoU** | confusion matrix | global intersection / global union |
| **Dice** | confusion matrix | mean Dice coefficient |
| **Confusion Matrix** | sklearn | NUM_LABELS × NUM_LABELS |

> **Note on Precision/Recall/F1 for segmentation**: sklearn treats each pixel
> as a sample. We flatten predictions and targets and use `macro` averaging so
> every class contributes equally regardless of pixel frequency.

In [ ]:
class SegMetrics:
    """
    Accumulates predictions across batches and computes all metrics at once.

    Stores:
      - confusion matrix (for IoU, Dice, per-class stats)
      - flat arrays of preds + targets (for sklearn P/R/F1)
    """

    def __init__(self, num_classes, ignore_index=-100, max_pixels=5_000_000):
        """
        num_classes  : number of label classes
        ignore_index : label value to exclude from metrics
        max_pixels   : cap on flat arrays to avoid OOM (randomly sampled)
        """
        self.K            = num_classes
        self.ignore_index = ignore_index
        self.max_pixels   = max_pixels
        self.reset()

    def reset(self):
        self.cm      = torch.zeros(self.K, self.K, dtype=torch.int64)
        self._preds  = []
        self._labels = []

    @torch.no_grad()
    def update(self, logits, targets):
        """
        logits  : (B, C, H, W)  raw model output
        targets : (B, H, W)     ground truth label indices
        """
        # Upsample if SegFormer returned at 1/4 resolution
        if logits.shape[-2:] != targets.shape[-2:]:
            logits = F.interpolate(
                logits.float(), size=targets.shape[-2:],
                mode='bilinear', align_corners=False
            )
        preds   = logits.argmax(dim=1).cpu()   # (B, H, W)
        targets = targets.cpu()

        valid   = (targets != self.ignore_index)
        p_flat  = preds[valid].numpy().astype(np.int32)
        t_flat  = targets[valid].numpy().astype(np.int32)

        # Update confusion matrix
        combined = self.K * t_flat + p_flat
        counts   = np.bincount(combined, minlength=self.K ** 2)
        self.cm += torch.tensor(counts.reshape(self.K, self.K), dtype=torch.int64)

        # Store flat arrays (capped to avoid OOM)
        if len(self._preds) < self.max_pixels:
            self._preds.append(p_flat)
            self._labels.append(t_flat)

    def compute(self):
        """
        Returns dict with all metrics.
        """
        cm  = self.cm.float()
        tp  = torch.diag(cm)
        fp  = cm.sum(dim=0) - tp
        fn  = cm.sum(dim=1) - tp

        # ── IoU & Dice per class ──────────────────────────────
        iou_per  = tp / (tp + fp + fn + 1e-6)
        dice_per = 2 * tp / (2 * tp + fp + fn + 1e-6)
        present  = cm.sum(dim=1) > 0   # classes that appear in GT

        miou  = iou_per[present].mean().item()
        oiou  = (tp.sum() / (tp.sum() + fp.sum() + fn.sum() + 1e-6)).item()
        mdice = dice_per[present].mean().item()

        # ── Pixel accuracy ────────────────────────────────────
        pixel_acc = (tp.sum() / (cm.sum() + 1e-6)).item()

        # ── sklearn Precision / Recall / F1 ──────────────────
        # Flatten accumulated arrays
        if self._preds:
            all_preds  = np.concatenate(self._preds)
            all_labels = np.concatenate(self._labels)
            # Use zero_division=0 to silence warnings for absent classes
            precision  = precision_score(all_labels, all_preds,
                                         average='macro', zero_division=0)
            recall     = recall_score(all_labels, all_preds,
                                      average='macro', zero_division=0)
            f1         = f1_score(all_labels, all_preds,
                                  average='macro', zero_division=0)
        else:
            precision = recall = f1 = 0.0

        return {
            'pixel_acc'    : pixel_acc,
            'precision'    : precision,
            'recall'       : recall,
            'f1'           : f1,
            'mIoU'         : miou,
            'oIoU'         : oiou,
            'Dice'         : mdice,
            'iou_per_class': iou_per.tolist(),
        }

    def compute_confusion_matrix(self):
        """Return the raw NUM_LABELS×NUM_LABELS confusion matrix as numpy."""
        return self.cm.numpy()


print('SegMetrics class ready.')

---
## 17. Early Stopping

Stops training when validation mIoU does not improve for `patience` epochs.  
`min_delta` is the minimum improvement that counts as a new best.  
When triggered, it restores the best model weights automatically.

In [ ]:
class EarlyStopping:
    """
    Monitors a metric (higher = better) and stops training if it
    fails to improve for `patience` consecutive epochs.

    Usage in training loop:
        es = EarlyStopping(patience=7, min_delta=0.001)
        if es.step(val_miou, model):
            print('Early stopping triggered!')
            break
        es.restore(model)   # load best weights after loop ends
    """

    def __init__(self, patience=7, min_delta=0.001, verbose=True):
        self.patience   = patience
        self.min_delta  = min_delta
        self.verbose    = verbose
        self.best_score = None
        self.counter    = 0
        self.best_state = None   # deepcopy of best model weights
        self.triggered  = False

    def step(self, score: float, model: nn.Module) -> bool:
        """
        Call after each validation epoch.
        Returns True if training should stop.
        """
        import copy
        if self.best_score is None or score > self.best_score + self.min_delta:
            self.best_score = score
            self.counter    = 0
            # Save a CPU copy of the best weights (no extra GPU memory)
            self.best_state = copy.deepcopy(
                {k: v.cpu() for k, v in model.state_dict().items()}
            )
            if self.verbose:
                print(f'  [EarlyStopping] New best: {score:.4f}  (patience counter reset)')
        else:
            self.counter += 1
            if self.verbose:
                print(f'  [EarlyStopping] No improvement. Counter: {self.counter}/{self.patience}')
            if self.counter >= self.patience:
                self.triggered = True
                return True
        return False

    def restore(self, model: nn.Module):
        """Load the best weights back into the model."""
        if self.best_state is not None:
            model.load_state_dict(
                {k: v.to(next(model.parameters()).device)
                 for k, v in self.best_state.items()}
            )
            print(f'  [EarlyStopping] Best weights restored (best mIoU={self.best_score:.4f})')


print('EarlyStopping class ready.')

---
## 18. Train / Evaluate Functions

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer,
                    onecycle_scheduler, scaler, device):
    """
    One epoch of FP16 training.
    onecycle_scheduler.step() is called EVERY BATCH (required for OneCycleLR).
    Returns: metrics dict with loss, pixel_acc, precision, recall, f1, mIoU.
    """
    model.train()
    metrics    = SegMetrics(NUM_LABELS)
    total_loss = 0.0

    for batch in loader:
        imgs    = batch['pixel_values'].to(device, non_blocking=True)
        targets = batch['labels'].to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with autocast():
            outputs = model(pixel_values=imgs)
            # SegFormer outputs (B, C, H/4, W/4) — upsample to full mask size
            logits  = F.interpolate(
                outputs.logits.float(),
                size=targets.shape[-2:],
                mode='bilinear', align_corners=False
            )
            loss = criterion(logits, targets)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        # OneCycleLR must step every batch, not every epoch
        onecycle_scheduler.step()

        total_loss += loss.item()
        metrics.update(logits.detach(), targets)

    stats         = metrics.compute()
    stats['loss'] = total_loss / len(loader)
    return stats


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    """
    Full evaluation pass. Returns metrics dict + confusion matrix.
    """
    model.eval()
    metrics    = SegMetrics(NUM_LABELS)
    total_loss = 0.0

    for batch in loader:
        imgs    = batch['pixel_values'].to(device, non_blocking=True)
        targets = batch['labels'].to(device, non_blocking=True)

        with autocast():
            outputs = model(pixel_values=imgs)
            logits  = F.interpolate(
                outputs.logits.float(),
                size=targets.shape[-2:],
                mode='bilinear', align_corners=False
            )
            loss = criterion(logits, targets)

        total_loss += loss.item()
        metrics.update(logits, targets)

    stats         = metrics.compute()
    stats['loss'] = total_loss / len(loader)
    stats['_cm']  = metrics.compute_confusion_matrix()   # kept for final plot
    return stats


print('train_one_epoch() and evaluate() defined.')

---
## 19. Hyperparameter Tuning with Optuna

**TPE sampler** (Bayesian) is much more sample-efficient than grid/random search.

Tuned parameters:
- `lr` — backbone LR (head LR = 10×)
- `weight_decay`
- `dice_weight` — Dice contribution in combined loss
- `focal_gamma` — focal loss focusing strength

Each trial trains 3 epochs on a 500-sample subset for speed.

In [ ]:
TUNE_EPOCHS = 3
TUNE_SUBSET = 500
N_TRIALS    = 15


def optuna_objective(trial):
    lr           = trial.suggest_float('lr',           1e-5, 5e-4, log=True)
    weight_decay = trial.suggest_float('weight_decay', 1e-5, 1e-2, log=True)
    dice_weight  = trial.suggest_float('dice_weight',  0.1,  1.0)
    focal_gamma  = trial.suggest_float('focal_gamma',  0.5,  4.0)

    t_model = build_segformer().to(DEVICE)

    bb_params   = [p for n, p in t_model.named_parameters() if 'decode_head' not in n]
    head_params = [p for n, p in t_model.named_parameters() if 'decode_head' in n]
    optimizer   = torch.optim.AdamW(
        [{'params': bb_params,   'lr': lr * 0.1},
         {'params': head_params, 'lr': lr}],
        weight_decay=weight_decay
    )

    indices  = np.random.choice(len(train_ds), TUNE_SUBSET, replace=False)
    t_loader = DataLoader(
        torch.utils.data.Subset(train_ds, indices),
        batch_size=BATCH_SIZE, shuffle=True, num_workers=2, drop_last=True
    )
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer,
        max_lr=[lr * 0.1, lr],
        total_steps=len(t_loader) * TUNE_EPOCHS,
        pct_start=0.3,
    )
    t_crit  = CombinedLoss(CLASS_WEIGHTS, dice_weight=dice_weight, gamma=focal_gamma)
    scaler  = GradScaler()

    for _ in range(TUNE_EPOCHS):
        train_one_epoch(t_model, t_loader, t_crit, optimizer, scheduler, scaler, DEVICE)

    val_stats = evaluate(t_model, val_loader, t_crit, DEVICE)
    del t_model
    torch.cuda.empty_cache()
    return val_stats['mIoU']


study = optuna.create_study(
    direction='maximize',
    sampler=TPESampler(seed=SEED),
    study_name='rrsisd_segformer'
)

print(f'Running {N_TRIALS} Optuna trials ({TUNE_EPOCHS} epochs each)...')
study.optimize(optuna_objective, n_trials=N_TRIALS, show_progress_bar=True)

best = study.best_params
print('\n=== Best Hyperparameters ===')
for k, v in best.items():
    print(f'  {k:<20s}: {v:.6f}')
print(f'  Best val mIoU  : {study.best_value:.4f}')

---
## 20. Full Training with Best Hyperparameters

### LR Scheduler strategy
We use **two complementary schedulers**:

1. **OneCycleLR** (per-batch) — aggressive warmup → cosine anneal.  
   Best for fine-tuning transformers from pretrained weights.  
   Steps every batch inside `train_one_epoch()`.

2. **ReduceLROnPlateau** (per-epoch) — halves LR when val mIoU plateaus.  
   Acts as a safety net if OneCycleLR exhausts its budget early.  
   Steps once per epoch after validation.

### Early stopping
Patience = 7 epochs. Restores best weights after triggering.

In [ ]:
# ── Hyperparameters (Optuna best, or override manually) ───────
LR           = best.get('lr',           6e-5)
WEIGHT_DECAY = best.get('weight_decay', 1e-4)
DICE_WEIGHT  = best.get('dice_weight',  0.5)
FOCAL_GAMMA  = best.get('focal_gamma',  2.0)
NUM_EPOCHS   = 50    # early stopping will stop earlier if needed
ES_PATIENCE  = 7     # early stopping patience (epochs)

# ── Build model & optimiser ───────────────────────────────────
model = build_segformer().to(DEVICE)

bb_params   = [p for n, p in model.named_parameters() if 'decode_head' not in n]
head_params = [p for n, p in model.named_parameters() if 'decode_head' in n]

# Differential learning rates:
#   backbone = LR/10 (pretrained, slow update preserves features)
#   head     = LR    (new classification head, fast update)
optimizer = torch.optim.AdamW(
    [{'params': bb_params,   'lr': LR * 0.1},
     {'params': head_params, 'lr': LR}],
    weight_decay=WEIGHT_DECAY
)

# ── Scheduler 1: OneCycleLR (per-batch) ───────────────────────
total_steps   = len(train_loader) * NUM_EPOCHS
onecycle_sch  = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=[LR * 0.1, LR],
    total_steps=total_steps,
    pct_start=0.1,          # 10% warmup
    div_factor=25,           # start LR = max_lr / 25
    final_div_factor=1e4,    # end LR   = max_lr / 10000
)

# ── Scheduler 2: ReduceLROnPlateau (per-epoch, safety net) ────
plateau_sch = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='max',       # monitoring mIoU (higher = better)
    factor=0.5,       # halve LR on plateau
    patience=3,       # wait 3 epochs before reducing
    min_lr=1e-7,
    verbose=True
)

criterion     = CombinedLoss(CLASS_WEIGHTS, dice_weight=DICE_WEIGHT, gamma=FOCAL_GAMMA)
scaler        = GradScaler()
early_stop    = EarlyStopping(patience=ES_PATIENCE, min_delta=0.001, verbose=True)
best_ckpt_path = os.path.join(OUTPUT_DIR, 'best_model.pt')

print(f'Config:')
print(f'  IMG_SIZE       : {IMG_SIZE}×{IMG_SIZE}')
print(f'  Backbone LR    : {LR * 0.1:.2e}')
print(f'  Head LR        : {LR:.2e}')
print(f'  Weight decay   : {WEIGHT_DECAY:.2e}')
print(f'  Dice weight    : {DICE_WEIGHT:.2f}')
print(f'  Focal gamma    : {FOCAL_GAMMA:.2f}')
print(f'  Max epochs     : {NUM_EPOCHS}')
print(f'  Early stopping : patience={ES_PATIENCE}')

In [ ]:
# ── History dict — stores every epoch's metrics for plotting ──
history = {
    'train_loss': [], 'val_loss': [],
    'train_acc':  [], 'val_acc':  [],
    'train_miou': [], 'val_miou': [],
    'train_f1':   [], 'val_f1':   [],
    'epoch_time': [],             # seconds per epoch
    'lr_backbone': [], 'lr_head': []
}

total_train_start = time.time()

for epoch in range(1, NUM_EPOCHS + 1):
    epoch_start = time.time()

    # ── Train ─────────────────────────────────────────────────
    train_stats = train_one_epoch(
        model, train_loader, criterion,
        optimizer, onecycle_sch, scaler, DEVICE
    )

    # ── Validate ──────────────────────────────────────────────
    val_stats = evaluate(model, val_loader, criterion, DEVICE)

    # ── ReduceLROnPlateau steps per epoch on val mIoU ─────────
    plateau_sch.step(val_stats['mIoU'])

    epoch_time = time.time() - epoch_start

    # ── Record history ────────────────────────────────────────
    history['train_loss'].append(train_stats['loss'])
    history['val_loss'].append(val_stats['loss'])
    history['train_acc'].append(train_stats['pixel_acc'])
    history['val_acc'].append(val_stats['pixel_acc'])
    history['train_miou'].append(train_stats['mIoU'])
    history['val_miou'].append(val_stats['mIoU'])
    history['train_f1'].append(train_stats['f1'])
    history['val_f1'].append(val_stats['f1'])
    history['epoch_time'].append(epoch_time)
    history['lr_backbone'].append(optimizer.param_groups[0]['lr'])
    history['lr_head'].append(optimizer.param_groups[1]['lr'])

    # ── Print epoch summary ────────────────────────────────────
    print(
        f'Epoch {epoch:3d}/{NUM_EPOCHS} | '
        f'loss {train_stats["loss"]:.4f}/{val_stats["loss"]:.4f} | '
        f'acc {train_stats["pixel_acc"]:.4f}/{val_stats["pixel_acc"]:.4f} | '
        f'mIoU {train_stats["mIoU"]:.4f}/{val_stats["mIoU"]:.4f} | '
        f'F1 {train_stats["f1"]:.4f}/{val_stats["f1"]:.4f} | '
        f'{epoch_time:.0f}s'
    )

    # ── Save best checkpoint ──────────────────────────────────
    if early_stop.best_score is None or val_stats['mIoU'] > early_stop.best_score:
        torch.save({
            'epoch':       epoch,
            'model_state': model.state_dict(),
            'optim_state': optimizer.state_dict(),
            'val_miou':    val_stats['mIoU'],
            'hparams':     best,
        }, best_ckpt_path)

    # ── Early stopping check ──────────────────────────────────
    if early_stop.step(val_stats['mIoU'], model):
        print(f'\nEarly stopping triggered at epoch {epoch}.')
        break

# Restore best weights after training ends (whether early-stopped or not)
early_stop.restore(model)

total_train_time = time.time() - total_train_start
avg_epoch_time   = np.mean(history['epoch_time'])

print(f'\nTraining complete.')
print(f'  Total training time   : {total_train_time / 60:.1f} min')
print(f'  Average time / epoch  : {avg_epoch_time:.1f} s')
print(f'  Best val mIoU         : {early_stop.best_score:.4f}')

---
## 21. Training Curves — Loss, Accuracy, mIoU, F1, LR

In [ ]:
def plot_training_curves(history, save_path=None):
    """
    Plot 5-panel training dashboard:
      1. Loss (train + val)
      2. Pixel Accuracy (train + val)
      3. mIoU (train + val)
      4. F1 Score (train + val)
      5. Learning Rate (backbone + head)
    """
    epochs = range(1, len(history['train_loss']) + 1)

    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    fig.suptitle('Training Dashboard — RRSIS-D SegFormer-B2 (480×480)',
                 fontsize=14, fontweight='bold', y=1.01)

    # ── 1. Loss ───────────────────────────────────────────────
    ax = axes[0, 0]
    ax.plot(epochs, history['train_loss'], 'b-o', ms=3, label='Train')
    ax.plot(epochs, history['val_loss'],   'r-o', ms=3, label='Val')
    ax.set_title('Loss (Focal + Dice)')
    ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
    ax.legend(); ax.grid(alpha=0.3)

    # ── 2. Pixel Accuracy ─────────────────────────────────────
    ax = axes[0, 1]
    ax.plot(epochs, [v * 100 for v in history['train_acc']], 'b-o', ms=3, label='Train')
    ax.plot(epochs, [v * 100 for v in history['val_acc']],   'r-o', ms=3, label='Val')
    ax.set_title('Pixel Accuracy (%)')
    ax.set_xlabel('Epoch'); ax.set_ylabel('Accuracy (%)')
    ax.legend(); ax.grid(alpha=0.3)

    # ── 3. mIoU ───────────────────────────────────────────────
    ax = axes[0, 2]
    ax.plot(epochs, [v * 100 for v in history['train_miou']], 'b-o', ms=3, label='Train')
    ax.plot(epochs, [v * 100 for v in history['val_miou']],   'r-o', ms=3, label='Val')
    ax.set_title('Mean IoU (%)')
    ax.set_xlabel('Epoch'); ax.set_ylabel('mIoU (%)')
    ax.legend(); ax.grid(alpha=0.3)

    # ── 4. F1 Score ───────────────────────────────────────────
    ax = axes[1, 0]
    ax.plot(epochs, [v * 100 for v in history['train_f1']], 'b-o', ms=3, label='Train')
    ax.plot(epochs, [v * 100 for v in history['val_f1']],   'r-o', ms=3, label='Val')
    ax.set_title('F1 Score — macro (%) ')
    ax.set_xlabel('Epoch'); ax.set_ylabel('F1 (%)')
    ax.legend(); ax.grid(alpha=0.3)

    # ── 5. Learning Rate ──────────────────────────────────────
    ax = axes[1, 1]
    ax.semilogy(epochs, history['lr_backbone'], 'g-', label='Backbone LR')
    ax.semilogy(epochs, history['lr_head'],     'm-', label='Head LR')
    ax.set_title('Learning Rate Schedule')
    ax.set_xlabel('Epoch'); ax.set_ylabel('LR (log scale)')
    ax.legend(); ax.grid(alpha=0.3, which='both')

    # ── 6. Epoch Time ─────────────────────────────────────────
    ax = axes[1, 2]
    ax.bar(epochs, history['epoch_time'], color='steelblue', alpha=0.7)
    ax.axhline(np.mean(history['epoch_time']), color='red',
               linestyle='--', label=f'Mean={np.mean(history["epoch_time"]):.0f}s')
    ax.set_title('Training Time per Epoch')
    ax.set_xlabel('Epoch'); ax.set_ylabel('Seconds')
    ax.legend(); ax.grid(alpha=0.3, axis='y')

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f'Saved to {save_path}')
    plt.show()


plot_training_curves(
    history,
    save_path=os.path.join(OUTPUT_DIR, 'training_curves.png')
)

---
## 22. Test Set Evaluation — All Metrics

In [ ]:
# Load best checkpoint
ckpt = torch.load(best_ckpt_path, map_location=DEVICE)
model.load_state_dict(ckpt['model_state'])
print(f'Loaded best checkpoint from epoch {ckpt["epoch"]}  '
      f'(val mIoU={ckpt["val_miou"]:.4f})\n')

test_stats = evaluate(model, test_loader, criterion, DEVICE)

print('=' * 55)
print(' TEST SET RESULTS')
print('=' * 55)
print(f'  Loss            : {test_stats["loss"]:.4f}')
print(f'  Pixel Accuracy  : {test_stats["pixel_acc"] * 100:.2f}%')
print(f'  Precision (mac) : {test_stats["precision"] * 100:.2f}%')
print(f'  Recall (macro)  : {test_stats["recall"] * 100:.2f}%')
print(f'  F1 (macro)      : {test_stats["f1"] * 100:.2f}%')
print(f'  mIoU            : {test_stats["mIoU"] * 100:.2f}%')
print(f'  oIoU            : {test_stats["oIoU"] * 100:.2f}%')
print(f'  Dice (mean)     : {test_stats["Dice"] * 100:.2f}%')
print('=' * 55)

print(f'\nModel info:')
print(f'  Total params      : {total_params:,}')
print(f'  Trainable params  : {trainable_params:,}')
print(f'  Model size        : {model_size_mb:.1f} MB')
print(f'  Total train time  : {total_train_time / 60:.1f} min')
print(f'  Avg epoch time    : {avg_epoch_time:.1f} s')
print(f'  Epochs run        : {len(history["train_loss"])}')

print('\nPer-class IoU:')
print(f'  {"ID":<4} {"Class":<32} {"IoU":>6}')
print('  ' + '-' * 46)
iou_list = test_stats['iou_per_class']
for idx in range(NUM_LABELS):
    print(f'  {idx:<4} {id2label[idx]:<32} {iou_list[idx] * 100:>5.2f}%')

---
## 23. Confusion Matrix

The confusion matrix shows how often each true class (row) is predicted
as each other class (column). The diagonal = correct predictions.  
Off-diagonal hotspots reveal which classes the model confuses most.

In [ ]:
def plot_confusion_matrix(cm, labels, save_path=None, normalise=True):
    """
    Plot a seaborn heatmap of the confusion matrix.

    cm        : (NUM_LABELS, NUM_LABELS) numpy int array
    labels    : list of class names
    normalise : if True, show row-normalised percentages (recommended
                for imbalanced datasets — prevents large classes from
                dominating the colour scale)
    """
    if normalise:
        row_sums = cm.sum(axis=1, keepdims=True)
        row_sums = np.where(row_sums == 0, 1, row_sums)   # avoid divide-by-zero
        cm_plot  = cm.astype(float) / row_sums
        fmt      = '.2f'
        title    = 'Confusion Matrix (row-normalised)'
    else:
        cm_plot  = cm
        fmt      = 'd'
        title    = 'Confusion Matrix (raw counts)'

    short_labels = [l[:12] for l in labels]   # truncate long class names

    fig, ax = plt.subplots(figsize=(18, 15))
    sns.heatmap(
        cm_plot,
        annot=True, fmt=fmt, cmap='Blues',
        xticklabels=short_labels,
        yticklabels=short_labels,
        linewidths=0.4, linecolor='lightgrey',
        ax=ax
    )
    ax.set_xlabel('Predicted class', fontsize=11)
    ax.set_ylabel('True class',      fontsize=11)
    ax.set_title(title,              fontsize=13, fontweight='bold')
    plt.xticks(rotation=45, ha='right', fontsize=8)
    plt.yticks(rotation=0,              fontsize=8)
    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f'Confusion matrix saved to {save_path}')
    plt.show()


# Plot normalised confusion matrix
plot_confusion_matrix(
    test_stats['_cm'],
    ALL_LABELS,
    save_path=os.path.join(OUTPUT_DIR, 'confusion_matrix.png'),
    normalise=True
)

# Plot raw-count confusion matrix (optional — comment out if too cluttered)
plot_confusion_matrix(
    test_stats['_cm'],
    ALL_LABELS,
    save_path=os.path.join(OUTPUT_DIR, 'confusion_matrix_raw.png'),
    normalise=False
)

---
## 24. Per-Class Metrics Bar Chart

In [ ]:
def plot_per_class_iou(iou_list, labels, save_path=None):
    """Horizontal bar chart of per-class IoU, sorted descending."""
    values = [iou_list[i] * 100 for i in range(len(labels))]
    pairs  = sorted(zip(labels, values), key=lambda x: x[1], reverse=True)
    names, vals = zip(*pairs)

    colours = ['#2196F3' if v >= 50 else '#FF9800' if v >= 25 else '#F44336'
               for v in vals]

    fig, ax = plt.subplots(figsize=(10, 9))
    bars = ax.barh(range(len(names)), vals, color=colours, edgecolor='white')
    ax.set_yticks(range(len(names)))
    ax.set_yticklabels(names, fontsize=9)
    ax.set_xlabel('IoU (%)')
    ax.set_title('Per-class IoU on Test Set', fontsize=12, fontweight='bold')
    ax.axvline(np.mean(vals), color='black', linestyle='--',
               label=f'Mean mIoU={np.mean(vals):.1f}%')
    ax.legend(fontsize=9)
    ax.grid(axis='x', alpha=0.3)

    # Annotate bar values
    for bar, v in zip(bars, vals):
        ax.text(v + 0.5, bar.get_y() + bar.get_height() / 2,
                f'{v:.1f}%', va='center', fontsize=8)

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()


plot_per_class_iou(
    test_stats['iou_per_class'],
    ALL_LABELS,
    save_path=os.path.join(OUTPUT_DIR, 'per_class_iou.png')
)

---
## 25. Full Metrics Summary Table

In [ ]:
# Print a clean markdown-style summary table
summary = {
    'Metric'                : [
        'Pixel Accuracy', 'Precision (macro)', 'Recall (macro)',
        'F1 Score (macro)', 'mIoU', 'oIoU', 'Dice (mean)',
        'Test Loss', '---',
        'Total Parameters', 'Trainable Parameters',
        'Model Size', 'Total Training Time', 'Avg Epoch Time',
        'Epochs Run (max)', 'Image Size'
    ],
    'Value': [
        f"{test_stats['pixel_acc'] * 100:.2f}%",
        f"{test_stats['precision'] * 100:.2f}%",
        f"{test_stats['recall'] * 100:.2f}%",
        f"{test_stats['f1'] * 100:.2f}%",
        f"{test_stats['mIoU'] * 100:.2f}%",
        f"{test_stats['oIoU'] * 100:.2f}%",
        f"{test_stats['Dice'] * 100:.2f}%",
        f"{test_stats['loss']:.4f}",
        '---',
        f'{total_params:,}',
        f'{trainable_params:,}',
        f'{model_size_mb:.1f} MB',
        f'{total_train_time / 60:.1f} min',
        f'{avg_epoch_time:.1f} s',
        f'{len(history["train_loss"])} / {NUM_EPOCHS}',
        f'{IMG_SIZE}×{IMG_SIZE}'
    ]
}

print(f'{"Metric":<28}  {"Value"}')
print('-' * 48)
for m, v in zip(summary['Metric'], summary['Value']):
    if m == '---':
        print('-' * 48)
    else:
        print(f'{m:<28}  {v}')

# Save summary as JSON
json_summary = {
    'test_metrics' : {k: v for k, v in test_stats.items() if k != '_cm'},
    'model_info'   : {
        'total_params'       : total_params,
        'trainable_params'   : trainable_params,
        'model_size_mb'      : model_size_mb,
        'img_size'           : IMG_SIZE,
    },
    'training_info': {
        'total_train_time_s' : total_train_time,
        'avg_epoch_time_s'   : avg_epoch_time,
        'epochs_run'         : len(history['train_loss']),
        'early_stopped'      : early_stop.triggered,
    },
    'hparams': best,
    'history': history
}
with open(os.path.join(OUTPUT_DIR, 'results_summary.json'), 'w') as f:
    json.dump(json_summary, f, indent=2)
print(f'\nResults saved to {os.path.join(OUTPUT_DIR, "results_summary.json")}')

---
## 26. Qualitative Visualisation

In [ ]:
@torch.no_grad()
def visualise_predictions(model, dataset, n=4, device=DEVICE):
    """
    Display n test images with ground-truth and predicted masks side-by-side.
    Pixel accuracy and present classes are shown per image.
    """
    model.eval()
    indices = random.sample(range(len(dataset)), n)

    fig, axes = plt.subplots(n, 3, figsize=(14, 4 * n))
    if n == 1:
        axes = axes[np.newaxis]

    for row, idx in enumerate(indices):
        sample    = dataset[idx]
        img_t     = sample['pixel_values'].unsqueeze(0).to(device)
        gt_mask   = sample['labels'].numpy()

        output    = model(pixel_values=img_t)
        logits    = F.interpolate(
            output.logits, size=gt_mask.shape,
            mode='bilinear', align_corners=False
        )
        pred_mask = logits.argmax(dim=1).squeeze().cpu().numpy()
        img_np    = denormalise(sample['pixel_values']).permute(1, 2, 0).numpy()
        px_acc    = np.mean(gt_mask == pred_mask)

        axes[row, 0].imshow(img_np)
        axes[row, 0].set_title('Image', fontsize=9)
        axes[row, 0].axis('off')

        axes[row, 1].imshow(gt_mask, cmap=CMAP, vmin=0, vmax=NUM_LABELS - 1)
        gt_classes = ', '.join(id2label[u][:8] for u in np.unique(gt_mask) if u > 0)[:40]
        axes[row, 1].set_title(f'GT: {gt_classes}', fontsize=7)
        axes[row, 1].axis('off')

        axes[row, 2].imshow(pred_mask, cmap=CMAP, vmin=0, vmax=NUM_LABELS - 1)
        axes[row, 2].set_title(f'Prediction  (px acc={px_acc:.2f})', fontsize=8)
        axes[row, 2].axis('off')

    patches = [mpatches.Patch(color=CMAP(i / NUM_LABELS), label=id2label[i])
               for i in range(NUM_LABELS)]
    fig.legend(handles=patches, loc='lower center', ncol=7,
               fontsize=7, bbox_to_anchor=(0.5, -0.02))
    plt.suptitle('Test set predictions', fontsize=12, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'test_predictions.png'),
                dpi=150, bbox_inches='tight')
    plt.show()


visualise_predictions(model, test_ds, n=4)

---
## 27. Export Model (HuggingFace Format)

In [ ]:
HF_SAVE_DIR = os.path.join(OUTPUT_DIR, 'segformer_rrsisd_final')
os.makedirs(HF_SAVE_DIR, exist_ok=True)

model.save_pretrained(HF_SAVE_DIR)
SegformerImageProcessor.from_pretrained('nvidia/mit-b2').save_pretrained(HF_SAVE_DIR)

print(f'Model saved to: {HF_SAVE_DIR}')

# Quick reload test
reloaded = SegformerForSemanticSegmentation.from_pretrained(HF_SAVE_DIR)
print(f'Reload OK. Classes: {reloaded.config.num_labels}')

---
## 28. Inference on a Single Image

In [ ]:
@torch.no_grad()
def predict_single(image_path: str, model, device=DEVICE):
    """
    Run segmentation on any single image file and display result.
    Returns dict with 'mask' (H×W ndarray) and 'classes' (list of str).
    """
    model.eval()
    img = cv2.imread(image_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]

    t   = val_transforms(image=img)
    inp = t['image'].unsqueeze(0).to(device)

    output = model(pixel_values=inp)
    logits = F.interpolate(
        output.logits, size=(h, w),
        mode='bilinear', align_corners=False
    )
    pred_mask = logits.argmax(dim=1).squeeze().cpu().numpy()

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    axes[0].imshow(img);            axes[0].set_title('Input');      axes[0].axis('off')
    axes[1].imshow(pred_mask, cmap=CMAP, vmin=0, vmax=NUM_LABELS - 1)
    axes[1].set_title('Prediction'); axes[1].axis('off')
    unique  = np.unique(pred_mask)
    patches = [mpatches.Patch(color=CMAP(u / NUM_LABELS), label=id2label[u])
               for u in unique]
    plt.legend(handles=patches, loc='lower right', fontsize=8)
    plt.tight_layout()
    plt.show()

    return {'mask': pred_mask, 'classes': [id2label[u] for u in unique]}


# Usage:
# result = predict_single('/path/to/your/image.jpg', model)
# print(result['classes'])

---
## Next Steps

1. **Scale up** — increase `NUM_EPOCHS` to 100 and `BATCH_SIZE` to 16 on Colab Pro A100
2. **Stronger backbone** — swap `nvidia/mit-b2` → `nvidia/mit-b3` for +2–3% mIoU
3. **Polygon masks** — if your XMLs have polygon annotations, update `parse_xml_to_mask()`
   to use `cv2.fillPoly` for tighter pixel-level supervision
4. **Multi-scale test-time augmentation (TTA)** — average predictions at 3 scales
   (×0.75, ×1.0, ×1.25) for +0.5–1% mIoU without retraining
5. **Integrate into RMSIN** — replace the SimpleDecoding head with this SegFormer
   backbone, feeding its 4 feature maps into the PWAM/GNN fusion modules